<a href="https://colab.research.google.com/github/leeaain2027/AIFFEL_quest_eng/blob/main/NLP/NLP01/260227_project_NLP_to_LLM_kor_eng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. 데이터 준비

### 구글 드라이브 마운트

In [ ]:
# 구글 드라이브 마운트 & 소스파일 가져오기
from google.colab import drive
from IPython.display import clear_output, HTML, display
import ipywidgets as widgets
import os

drive.mount('/content/drive')
clear_output()

base_path = '/content/drive/MyDrive/Aiffel'
mypath = os.path.join(base_path, 'NLP/Project/LLM_1')

def info_msg(msg, style=1, width='450px'):
    # color = '#2ecc71' if style == 1 else '#e67e22'
    color = '#3498db' if style == 1 else '#e67e22'
    mark = '\u2714' if style == 1 else '\u26A0 Warning: '

    html_code = f"""
    <div style="
        background-color: {color};
        color: white;
        padding: 20px 20px;
        border-radius: 15px;
        width: {width};
        text-align: center;
        font-family: 'Malgun Gothic', sans-serif;
        font-weight: bold;
        font-size: 16px;

        margin: 12px 0;
    ">
        {mark} {msg}
    </div>
    """
    display(HTML(html_code))

print(mypath)
info_msg('Google Drive has mounted.', 1)

/content/drive/MyDrive/Aiffel/NLP/Project/LLM_1


### import

In [ ]:
!sudo apt update
!sudo apt-get install -y fonts-nanum
!pip install sentencepiece

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,115 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/un

In [ ]:
import pandas
import torch
import matplotlib

print(pandas.__version__)
print(torch.__version__)
print(matplotlib.__version__)

2.2.2
2.10.0+cu128
3.10.0


In [ ]:
# 1. 시스템 의존성 및 Mecab 관련 패키지 설치
!sudo apt-get install g++ openjdk-8-jdk python3-dev curl
!pip install konlpy
!pip install mecab-python3

# 2. Mecab-ko (형태소 분석기 엔진) 설치
!curl -sL https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh | bash

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
g++ is already the newest version (4:11.2.0-1ubuntu1).
g++ set to manually installed.
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  javascript-common libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0 libcurl4
  libcurl4-openssl-dev libgail-common libgail18 libgtk2.0-0 libgtk2.0-bin
  libgtk2.0-common libjs-sphinxdoc libjs-underscore librsvg2-common
  libxcomposite1 libxt-dev libxtst6 libxxf86dga1 openjdk-8-jdk-headless
  openjdk-8-jre openjdk-8-jre-headless python3.10-dev session-migration
  x11-utils
Suggested packages:
  apache2 | lighttpd | httpd libcurl4-doc libidn11-dev libldap2-dev
  librtmp-dev gvfs libxt-doc openjdk-8-demo openjdk-8-source visualvm
  libnss-mdns fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei f

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
import logging

import os
import re
import urllib.request
import zipfile
import sentencepiece as spm
import pandas as pd

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

from tqdm import tqdm
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

fontpath = "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"
fontprop = fm.FontProperties(fname=fontpath, size=12)
plt.rcParams["font.family"] = fontprop.get_name()

print(f"설정된 폰트: {fontprop.get_name()}")

2.10.0+cu128
설정된 폰트: NanumBarunGothic


### 데이터 다운로드


https://github.com/jungyeul/korean-parallel-corpora/blob/master/korean-english-news-v1/korean-english-park.train.tar.gz

In [ ]:
import tarfile

project_dir = os.path.expanduser("work/s2s_translation_project")
dataset_dir = os.path.join(project_dir, "datasets")
os.makedirs(dataset_dir, exist_ok=True)

file_path = os.path.join(project_dir, "korean-english-park.train.tar.gz")

if not os.path.exists(file_path):
    print("데이터 다운로드 중...")
    url = "https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz"
    urllib.request.urlretrieve(url, file_path)
    print("다운로드 완료!")

print("압축 해제 중...")
with tarfile.open(file_path, "r:gz") as tar:
    tar.extractall(path=dataset_dir)
print("압축 해제 완료!")

print("데이터셋 디렉토리:", os.listdir(dataset_dir))

데이터 다운로드 중...
다운로드 완료!
압축 해제 중...


/tmp/ipykernel_1959/3257015142.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=dataset_dir)


압축 해제 완료!
데이터셋 디렉토리: ['korean-english-park.train.en', 'korean-english-park.train.ko']


In [ ]:
import pandas as pd

file_en = os.path.join(dataset_dir, "korean-english-park.train.en")
file_ko = os.path.join(dataset_dir, "korean-english-park.train.ko")

# 한국어 파일 읽기
df_ko = pd.read_csv(file_ko,
                    sep="\t",          # 탭으로 구분하라고 시킴 (문장에 탭은 거의 없으니까요)
                    names=["ko"],      # 컬럼 이름 지정
                    header=None,       # 첫 줄을 제목으로 쓰지 않음
                    quoting=3,         # csv.QUOTE_NONE과 동일 (따옴표 무시)
                    on_bad_lines='skip', # 혹시나 에러 나는 줄은 일단 넘김
                    encoding='utf-8')

# 영어 파일 읽기
df_en = pd.read_csv(file_en,
                    sep="\t",
                    names=["en"],
                    header=None,
                    quoting=3,
                    encoding='utf-8')

# 길이 확인
print(len(df_ko))
print(len(df_en))

# 두 데이터 합치기
df = pd.concat([df_ko, df_en], axis=1)
df

94123
94123


,ko,en
0,"개인용 컴퓨터 사용의 상당 부분은 ""이것보다 뛰어날 수 있느냐?""","Much of personal computing is about ""can you t..."
1,모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하...,so a mention a few weeks ago about a rechargea...
2,그러나 이것은 또한 책상도 필요로 하지 않는다.,"Like all optical mice, But it also doesn't nee..."
3,"79.95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목, 팔, 그외에 어떤 부분...",uses gyroscopic sensors to control the cursor ...
4,정보 관리들은 동남 아시아에서의 선박들에 대한 많은 (테러) 계획들이 실패로 돌아갔...,Intelligence officials have revealed a spate o...
...,...,...
94118,“우리는 3월 8일 김승연 회장과 그의 아들이 보복폭행에 가담한 혐의를 찾기 위해 ...,””We are hoping to seize material evidence to ...
94119,월요일 술집 종업원 6명은 김회장과 아들에게 폭행을 당했음을 진술했다고 경찰은 말했다.,"” On Monday, police secured statements from si..."
94120,그러나 불충분한 증거 확보로 수사에 어려움이 있다.,But the lack of material evidence is making it...
94121,김회장과 그의 아들은 보복폭행 혐의를 강력히 부인하고 있다.,Kim and his son both deny the allegations.


In [ ]:
print(df_ko[1001:1005])
print(df_en[1001:1005])

                                                     ko
1001  일단의 자연 과학자들이 정글을 있는 그대로 두고 숲 속에서 나는 천연의 상품들을 수...
1002                    그들은 그 개념을 "유지할 수 있는 개발"이라고 부른다.
1003             이 경제 모델에서는 2.5에이커의 토지가 여러가지 용도로 사용되었다.
1004                           목재는 몇 백 달러를 더 벌어들일 뿐이었다.
                                                     en
1001  A group of natural scientists / have developed...
1002  They call the concept / "sustainable developme...
1003  / In the economic model, / two and a half acre...
1004  / Timber brought in only a few hundred dollars...


# 2. 데이터 전처리

### set을 이용한 중복 제거

In [ ]:
# 중복 제거 (병렬 쌍을 기준으로)
# 두 컬럼을 튜플의 리스트로 결합 (Zip 활용)

# zip은 ko[0]-en[0], ko[1]-en[1] 식으로 짝을 지어줍니다.
combined_pairs = list(zip(df_ko['ko'], df_en['en']))

# set을 사용하여 중복 제거
# (한국어문장, 영어문장) 튜플 자체가 하나의 원소가 되어 중복을 검사합니다.
unique_pairs = set(combined_pairs)

# 4. 결과 확인
print(f"중복 제거 전: {len(combined_pairs)}")
print(f"중복 제거 후: {len(unique_pairs)}")

# 5. 다시 데이터프레임으로 복구 (학습에 사용하기 편하도록)
df = pd.DataFrame(list(unique_pairs), columns=['ko', 'en'])

# 결과 확인
print(df.head())
print(df.tail())

중복 제거 전: 94123
중복 제거 후: 78968
                                                  ko  \
0  원문기사보기 ‘골프의 전설’ 세비 바예스테로스가 자신이 뇌종양으로 투병 중임을 확인했다.   
1            군과 반군 간의 휴전이 깨진 7월 중순 이래로 폭력사태가 증가해 왔다.   
2  FBI와 미 국토안보부 산하 비밀검찰국(US Secret Service)은 이번 사...   
3       다른 관련 범죄가 줄줄이 발견되어 가택침입 등 총 19건으로 기소됐기 때문이다.   
4  프랑스 오픈을 한 차례도 우승하지 못했던 페더러는 롤랑 가로스를 제패해야만 미국의 ...   

                                                  en  
0  Golf legend Seve Ballesteros has confirmed tha...  
1  The violence has been escalating since mid-Jul...  
2  The FBI and Secret Service refused comment on ...  
3  he has been charged with a total of 19 break-i...  
4  Federer, who has never won the French Open, ne...  
                                                      ko  \
78963  연구진은 개가 자신의 주인이 다른 개에게 애정을 보이는 것을 싫어하고 갓난아기나 애...   
78964                                알렉 볼드윈, 이혼관련 자서전 발간   
78965     우주왕복선 인데버호의 승무원들은 뒤늦은 추수감사절 음식을 먹게 될 것으로 보입니다.   
78966                                      7. 유원지 시설 보조원   
789

### 데이터 정제
- 불필요한 특수문자는 노이즈로 작용할 수 있음
- 문장에 시작 문자 <start>, 종료 문자 <end> 붙여주기  
    Decoder는 첫 입력으로 사용할 시작 토큰과 문장생성 종료를 알리는 끝 토큰이 반드시 필요하다.
    Encoder의 입력문장에서는 굳이 필요없음.

In [ ]:
# 데이터 정제
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()

    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)

    sentence = sentence.strip()

    return sentence

In [ ]:
# 2) 전처리 함수 재정의 (한글+영문 모두 처리)
#    - 한글용 정규식 추가
#    - 필요시 소문자화, 특수문자 제거 등
def preprocessing(text: str, is_korean: bool = False) -> str:
    text = str(text).strip()

    if is_korean:
        # 한글, 숫자, 기본 문장부호 외 제거
        # ㄱ-ㅎ가-힣, 0-9, a-zA-Z, 기본 문장 부호만 남기기
        text = re.sub(r"[^0-9a-zA-Zㄱ-ㅎ가-힣\s\.\,\!\?\-]", " ", text)
    else:
        # 영문: 소문자화, 알파벳/숫자/기본 문장부호만
        text = text.lower()
        text = re.sub(r"[^0-9a-z\s\.\,\!\?\-]", " ", text)

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Mecab 초기화
mecab = Mecab()  # 환경에 따라 dicpath 지정 필요할 수 있음


def tokenize_korean(text: str):
    # Mecab으로 형태소 단위 토큰화
    return mecab.morphs(text)

def tokenize_english(text: str):
    # 공백 기준 토큰화
    return text.split()

In [ ]:
# 3) <start>, <end> 토큰 추가 및 길이 필터링 (토큰 수 40 이하)
eng_corpus = []
kor_corpus = []

max_len = 40

for ko, en in cleaned_corpus:
    ko_p = preprocessing(ko, is_korean=True)
    en_p = preprocessing(en, is_korean=False)

    # 토큰화
    ko_tokens = tokenize_korean(ko_p)
    en_tokens = tokenize_english(en_p)

    # 토큰 수 필터링 (여기서는 소스/타겟 둘 다 40 이하로 제한)
    if len(ko_tokens) <= max_len and len(en_tokens) <= max_len:
        # 영어에 <start>, <end> 추가
        en_tokens = ["<start>"] + en_tokens + ["<end>"]
        eng_corpus.append(en_tokens)
        kor_corpus.append(ko_tokens)

print("총 문장 수:", len(eng_corpus), len(kor_corpus))

총 문장 수: 63744 63744


## Step 3. 데이터 토큰화 → 텐서 변환

In [ ]:
# 간단한 토크나이저 구현 (word2idx, idx2word)
class SimpleTokenizer:
    def __init__(self, min_freq: int = 1, max_vocab_size: int = 20000,
                 specials=None):
        if specials is None:
            specials = ["<pad>", "<unk>"]
        self.min_freq = min_freq
        self.max_vocab_size = max_vocab_size
        self.specials = specials
        self.word2idx = {}
        self.idx2word = []

    def fit(self, corpus):
        # corpus: List[List[str]]
        freq = {}
        for sent in corpus:
            for w in sent:
                freq[w] = freq.get(w, 0) + 1

        # 빈도 순 정렬
        sorted_words = sorted(freq.items(), key=lambda x: x[1], reverse=True)

        # 특수 토큰 먼저
        self.idx2word = list(self.specials)
        for i, w in enumerate(self.idx2word):
            self.word2idx[w] = i

        # 나머지 단어 추가
        for w, c in sorted_words:
            if c < self.min_freq:
                continue
            if len(self.idx2word) >= self.max_vocab_size:
                break
            self.word2idx[w] = len(self.idx2word)
            self.idx2word.append(w)

    def encode(self, tokens):
        # List[str] -> List[int]
        unk_idx = self.word2idx.get("<unk>", 1)
        return [self.word2idx.get(w, unk_idx) for w in tokens]

    def decode(self, ids):
        # List[int] -> List[str]
        return [self.idx2word[i] if 0 <= i < len(self.idx2word) else "<unk>" for i in ids]

    @property
    def vocab_size(self):
        return len(self.idx2word)

In [ ]:
# 토크나이저 학습 (단어 수 10,000 이상이 되도록 max_vocab_size를 충분히 크게)
src_tokenizer = SimpleTokenizer(min_freq=1, max_vocab_size=20000)
tgt_tokenizer = SimpleTokenizer(min_freq=1, max_vocab_size=20000,
                                specials=["<pad>", "<unk>", "<start>", "<end>"])

src_tokenizer.fit(kor_corpus)
tgt_tokenizer.fit(eng_corpus)

print("한국어 vocab size:", src_tokenizer.vocab_size)
print("영어 vocab size:", tgt_tokenizer.vocab_size)

한국어 vocab size: 20000
영어 vocab size: 20000


## Step 4. 모델 설계

In [ ]:
# Dataset / DataLoader 구성
class TranslationDataset(Dataset):
    def __init__(self, src_corpus, tgt_corpus, src_tokenizer, tgt_tokenizer,
                 max_len=40):
        self.src_corpus = src_corpus
        self.tgt_corpus = tgt_corpus
        self.src_tok = src_tokenizer
        self.tgt_tok = tgt_tokenizer
        self.max_len = max_len
        self.pad_idx = self.tgt_tok.word2idx["<pad>"]

    def __len__(self):
        return len(self.src_corpus)

    def __getitem__(self, idx):
        src_tokens = self.src_corpus[idx]
        tgt_tokens = self.tgt_corpus[idx]

        src_ids = self.src_tok.encode(src_tokens)
        tgt_ids = self.tgt_tok.encode(tgt_tokens)

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(
            tgt_ids, dtype=torch.long
        )


In [ ]:
def collate_fn(batch):
    # 동적 padding
    src_seqs, tgt_seqs = zip(*batch)
    src_lengths = [len(s) for s in src_seqs]
    tgt_lengths = [len(s) for s in tgt_seqs]

    max_src_len = max(src_lengths)
    max_tgt_len = max(tgt_lengths)

    pad_idx_src = src_tokenizer.word2idx["<pad>"] if "<pad>" in src_tokenizer.word2idx else 0
    pad_idx_tgt = tgt_tokenizer.word2idx["<pad>"]

    batch_src = []
    batch_tgt = []

    for s, t in zip(src_seqs, tgt_seqs):
        src_padded = torch.full((max_src_len,), pad_idx_src, dtype=torch.long)
        tgt_padded = torch.full((max_tgt_len,), pad_idx_tgt, dtype=torch.long)
        src_padded[: len(s)] = s
        tgt_padded[: len(t)] = t
        batch_src.append(src_padded)
        batch_tgt.append(tgt_padded)

    batch_src = torch.stack(batch_src, dim=0)
    batch_tgt = torch.stack(batch_tgt, dim=0)

    return batch_src, batch_tgt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 하이퍼파라미터 (실험으로 조정)
src_vocab_size = src_tokenizer.vocab_size
tgt_vocab_size = tgt_tokenizer.vocab_size
embed_size = 256
hidden_size = 512
num_layers = 1
dropout = 0.1

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1,
                 dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers=num_layers,
                          batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, T)
        embedded = self.dropout(self.embedding(x))  # (B, T, E)
        outputs, hidden = self.rnn(embedded)        # outputs: (B, T, 2H)
        # hidden: (2*num_layers, B, H)

        # 양방향 GRU의 마지막 hidden을 합쳐서 decoder 초기 hidden으로 변환
        h_forward = hidden[-2, :, :]  # (B, H)
        h_backward = hidden[-1, :, :]  # (B, H)
        h_cat = torch.cat((h_forward, h_backward), dim=1)  # (B, 2H)
        h_0 = torch.tanh(self.fc(h_cat)).unsqueeze(0)      # (1, B, H)
        return outputs, h_0

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, enc_hidden_size, dec_hidden_size):
        super().__init__()
        self.W1 = nn.Linear(enc_hidden_size * 2, dec_hidden_size)
        self.W2 = nn.Linear(dec_hidden_size, dec_hidden_size)
        self.V = nn.Linear(dec_hidden_size, 1)

    def forward(self, encoder_outputs, hidden):
        # encoder_outputs: (B, T, 2H)
        # hidden: (1, B, H) -> (B, H)
        hidden = hidden.permute(1, 0, 2)  # (B, 1, H)
        score = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))
        # score: (B, T, H)
        attention_weights = torch.softmax(self.V(score), dim=1)  # (B, T, 1)
        context = attention_weights * encoder_outputs  # (B, T, 2H)
        context = context.sum(dim=1)  # (B, 2H)
        return context, attention_weights.squeeze(-1)  # (B, 2H), (B, T)


In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, enc_hidden_size, dec_hidden_size,
                 num_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size + enc_hidden_size * 2, dec_hidden_size,
                          num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(dec_hidden_size, vocab_size)
        self.attention = BahdanauAttention(enc_hidden_size, dec_hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, encoder_outputs):
        # input: (B) - 이번 time step의 토큰
        # hidden: (1, B, H)
        # encoder_outputs: (B, T, 2H)
        input = input.unsqueeze(1)  # (B, 1)
        embedded = self.dropout(self.embedding(input))  # (B, 1, E)

        # attention
        context, attn_weights = self.attention(encoder_outputs, hidden)
        context = context.unsqueeze(1)  # (B, 1, 2H)

        rnn_input = torch.cat((embedded, context), dim=2)  # (B, 1, E+2H)
        output, hidden = self.rnn(rnn_input, hidden)       # output: (B, 1, H)
        prediction = self.fc_out(output.squeeze(1))        # (B, V)
        return prediction, hidden, attn_weights

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # src: (B, T_src), tgt: (B, T_tgt)
        batch_size = src.size(0)
        max_len = tgt.size(1)
        vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(batch_size, max_len, vocab_size, device=device)

        encoder_outputs, hidden = self.encoder(src)
        input = tgt[:, 0]  # <start> 토큰이어야 함

        for t in range(1, max_len):
            output, hidden, attn = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(dim=1)
            input = tgt[:, t] if teacher_force else top1

        return outputs

In [ ]:
encoder = Encoder(src_vocab_size, embed_size, hidden_size, num_layers, dropout)
decoder = Decoder(tgt_vocab_size, embed_size, hidden_size, hidden_size,
                  num_layers, dropout)

pad_idx = tgt_tokenizer.word2idx["<pad>"]
model = Seq2Seq(encoder, decoder, pad_idx).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Step5. 훈련

In [ ]:
save_dir = os.path.join(project_dir, "checkpoints")
os.makedirs(save_dir, exist_ok=True)

num_epochs = 20
early_stop_patience = 3  # validation이 없다면, 단순 loss 기준으로
best_loss = float("inf")
epochs_no_improve = 0

example_sentence = "한국의 경제 상황에 대해 설명해 주세요."  # 매 스텝 번역 예시
example_sentence = preprocessing(example_sentence, is_korean=True)
example_tokens = tokenize_korean(example_sentence)

In [ ]:
def translate_example(model, sentence_tokens, max_len=40):
    model.eval()
    with torch.no_grad():
        src_ids = src_tokenizer.encode(sentence_tokens)
        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)

        encoder_outputs, hidden = model.encoder(src_tensor)
        # 시작 토큰
        start_idx = tgt_tokenizer.word2idx["<start>"]
        end_idx = tgt_tokenizer.word2idx["<end>"]

        input = torch.tensor([start_idx], dtype=torch.long, device=device)
        outputs = []

        for _ in range(max_len):
            preds, hidden, attn = model.decoder(input, hidden, encoder_outputs)
            top1 = preds.argmax(dim=1)
            if top1.item() == end_idx:
                break
            outputs.append(top1.item())
            input = top1

        words = tgt_tokenizer.decode(outputs)
        # 불필요한 토큰 제거
        return " ".join(words)

In [ ]:
for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0
    for batch_idx, (src_batch, tgt_batch) in enumerate(train_loader):
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)

        optimizer.zero_grad()
        outputs = model(src_batch, tgt_batch, teacher_forcing_ratio=0.5)
        # outputs: (B, T, V), tgt_batch: (B, T)
        outputs_reshaped = outputs[:, 1:].reshape(-1, outputs.size(-1))
        tgt_reshaped = tgt_batch[:, 1:].reshape(-1)

        loss = criterion(outputs_reshaped, tgt_reshaped)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()

        # 매 스텝(또는 몇 스텝 간격)마다 예문 번역 확인 (여기선 간단히 batch 기준)
        if (batch_idx + 1) % 100 == 0:
            translated = translate_example(model, example_tokens)
            print(f"[Epoch {epoch} | Step {batch_idx+1}] 예문 번역:", translated)

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch} 평균 Loss: {avg_loss:.4f}")

    # 에포크별 모델 저장
    model_path = os.path.join(save_dir, f"seq2seq_epoch_{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": avg_loss,
        "src_tokenizer": src_tokenizer,
        "tgt_tokenizer": tgt_tokenizer,
    }, model_path)
    print(f"모델 저장: {model_path}")

    # 조기 종료 체크 (여기서는 train loss 기준, 실제로는 val loss를 쓰는 것이 좋음)
    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_no_improve = 0
        # best 모델도 별도 저장
        best_model_path = os.path.join(save_dir, "seq2seq_best.pt")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss,
            "src_tokenizer": src_tokenizer,
            "tgt_tokenizer": tgt_tokenizer,
        }, best_model_path)
        print(f"Best 모델 갱신: {best_model_path}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"조기 종료 발동! {epoch} 에포크에서 훈련 종료.")
            break

[Epoch 1 | Step 100] 예문 번역: the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the


RuntimeError: cudnn RNN backward can only be called in training mode

Teacher Forcing이 중요한 이유  
- 학습 초기에는 모델이 엉뚱한 단어를 출력. 이 단어를 다음 시점 입력으로 가져가면 문장이 엉망이 되어 학습 불가능
- 학습할 동안에는 모델이 무엇을 출력하건 다음 입력에 실제 정답을 강제로 넣어주는 기법